# Relaxation of a water molecule with QEpy and ASE

### To run this tutorial in google colab click the following link :

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/asesma-org/miniASESMA2026/blob/main/Week1/Day3/hands-on-aimd/hands-on_aimd.ipynb)

In [ ]:
## Uncomment the following lines for Colab. IMPORTANT NOTE: install first qepy and then dftpy
# !pip install qepy f90wrap==0.2.16

In [ ]:
# !python -m pip install "git+https://github.com/Quantum-MultiScale/DFTpy.git@dev"

In [1]:
import qepy
import numpy as np

In [2]:
from qepy.calculator import QEpyCalculator
import ase.io
from ase import units

In [ ]:
## Uncomment these lines for Colab
# additional_files = {
#     'O_ONCV_PBE-1.2.upf' : 'https://raw.githubusercontent.com/asesma-org/miniASESMA2026/main/Week1/Day3/hands-on-relax/O_ONCV_PBE-1.2.upf',
#     'H_ONCV_PBE-1.2.upf' : 'https://raw.githubusercontent.com/asesma-org/miniASESMA2026/main/Week1/Day3/hands-on-relax/H_ONCV_PBE-1.2.upf',
#     'qe_in.in': 'https://raw.githubusercontent.com/asesma-org/miniASESMA2026/main/Week1/Day3/hands-on-aimd/qe_in.in',
# }
# from dftpy.formats import download_files
# download_files(additional_files)

### We are providing PPs in this folder for ease of execution

In [3]:
from ase.build import molecule
atoms = molecule ("H2O")                # define the molecule\
atoms.set_cell(cell=np.identity(3)*7)   # set simulation cell (in Angstroms)
atoms.translate([3.5,3.5,3.5])          # translate molecule to center of cell (for convenience)

In [4]:
inputfile = 'qe_in.in'

In [6]:
scf_options = {}
scf_options['&electrons'] = {}
scf_options['&electrons']['conv_thr'] = 1e-7
scf_options['atomic_species'] = []
scf_options['atomic_species'].append('O  15.99  O_ONCV_PBE-1.2.upf') # PP provided
scf_options['atomic_species'].append('H  1.008  H_ONCV_PBE-1.2.upf') # PP provided
scf_options['k_points {gamma}'] = []

### Set up the QEpy calculator

In [7]:
atoms.calc = QEpyCalculator(inputfile=inputfile, qe_options=scf_options, atoms=atoms, logfile='tmp.out')

In [8]:
from ase.optimize import BFGS
dyn = BFGS(atoms)
dyn.run(fmax=0.05)

      Step     Time          Energy          fmax
BFGS:    0 04:16:49     -463.210691        0.657970
BFGS:    1 04:16:55     -463.214752        0.183542
BFGS:    2 04:16:59     -463.215288        0.049277


True

In [9]:
atoms.get_potential_energy()

-463.21528828003227

In [10]:
atoms.get_forces()

array([[-1.44279028e-07, -2.92194909e-04,  2.22044744e-02],
       [ 7.15787376e-08, -4.77457812e-02, -1.12245612e-02],
       [ 7.27002908e-08,  4.80379761e-02, -1.09799131e-02]])

In [11]:
from ase.io.trajectory import Trajectory
from ase import units
from ase.md.andersen import Andersen
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
T = 300
MaxwellBoltzmannDistribution(atoms, temperature_K = T, force_temp=True)
dyn = Andersen(atoms, 1.0 * units.fs, temperature_K = T, andersen_prob=0.02)

In [12]:
step = 0
interval = 1
traj = Trajectory("tmp.traj", "w", atoms)

In [16]:
def printenergy(a=atoms):
    global step, interval
    epot = a.get_potential_energy() / len(a)
    ekin = a.get_kinetic_energy() / len(a)
    print("Step={:<8d} Epot={:.5f} Ekin={:.5f} T={:.3f} Etot={:.5f}".format(
                step, epot, ekin, ekin / (1.5 * units.kB), epot + ekin), flush = True)
    step += interval

In [17]:
dyn.attach(printenergy, interval=1)
dyn.attach(traj.write, interval=1)
dyn.run(5)

Step=6        Epot=-154.40184 Ekin=0.01633 T=126.353 Etot=-154.38550
Step=8        Epot=-154.40318 Ekin=0.01765 T=136.562 Etot=-154.38553
Step=10       Epot=-154.40157 Ekin=0.01665 T=128.794 Etot=-154.38492
Step=12       Epot=-154.39837 Ekin=0.01412 T=109.253 Etot=-154.38425
Step=14       Epot=-154.39870 Ekin=0.01479 T=114.382 Etot=-154.38391


True